In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.covariance import EllipticEnvelope
from scipy.spatial import ConvexHull
from scipy.optimize import linprog

# Optional KMedoids
try:
    from sklearn_extra.cluster import KMedoids
    HAS_KMEDOIDS = True
except Exception:
    HAS_KMEDOIDS = False

# -----------------------------
# Data generation
# -----------------------------
def generate_points(n=40, seed=42):
    rng = np.random.default_rng(seed)
    # Two latent planes + noise
    A = rng.normal(loc=0.0, scale=1.0, size=(n//2, 3))
    B = rng.normal(loc=2.5, scale=1.0, size=(n - n//2, 3))
    X = np.vstack([A, B])
    # Light correlation
    X[:, 2] = 0.3 * X[:, 0] - 0.4 * X[:, 1] + rng.normal(0, 0.2, size=n)
    return X

# -----------------------------
# Distance-based clustering (2D)
# -----------------------------
def distance_based_clustering_2d(P2d, k):
    centroid = P2d.mean(axis=0)
    dists = np.linalg.norm(P2d - centroid, axis=1)
    order = np.argsort(dists)[::-1]  # farthest first
    n = len(P2d)
    group_size = n // k
    labels = np.empty(n, dtype=int)
    for i in range(k):
        start = i * group_size
        end = (i + 1) * group_size if i < k - 1 else n
        idx = order[start:end]
        labels[idx] = i
    return labels

# -----------------------------
# Carathéodory set (LP-based)
# -----------------------------
def caratheodory_set(v, P, r):
    # P: (n,d), v: (d,)
    n = P.shape[0]
    A_eq = np.vstack([P.T, np.ones(n)])
    b_eq = np.hstack([v, 1.0])
    c = np.zeros(n)
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=(0, None), method='highs')
    if not res.success:
        return np.array([], dtype=int)
    u = res.x
    tol = 1e-9
    support = np.where(u > tol)[0]
    # Sparsify until <= r+1
    P_sup = P[support]
    u_sup = u[support]
    while len(support) > r + 1:
        m = len(support)
        A = np.vstack([P_sup.T, np.ones(m)])
        _, s, Vt = np.linalg.svd(A, full_matrices=False)
        if s[-1] > 1e-7:
            break  # numerical stop
        alpha = Vt[-1]
        if not np.any(alpha > 0):
            alpha = -alpha
            if not np.any(alpha > 0):
                break
        t_candidates = u_sup[alpha > 0] / alpha[alpha > 0]
        if len(t_candidates) == 0:
            break
        t = np.min(t_candidates)
        u_sup = u_sup - t * alpha
        u_sup = np.clip(u_sup, 0, None)
        keep = u_sup > tol
        P_sup = P_sup[keep]
        support = support[keep]
        u_sup = u_sup[keep]
    return support

# -----------------------------
# MVEE via EllipticEnvelope (2D cluster)
# -----------------------------
def compute_mvee(P2d_cluster):
    clf = EllipticEnvelope(support_fraction=1.0, contamination=0.01, random_state=0)
    clf.fit(P2d_cluster)
    c = clf.location_
    Sigma = clf.covariance_
    # Ensure PSD
    vals, vecs = np.linalg.eigh(Sigma)
    vals = np.clip(vals, 1e-8, None)
    Sigma = (vecs @ np.diag(vals) @ vecs.T)
    return c, Sigma

def ellipse_poly(c, Sigma, num=200, scale=2.0):
    vals, vecs = np.linalg.eigh(Sigma)
    angles = np.linspace(0, 2*np.pi, num)
    pts = []
    for theta in angles:
        unit = np.array([np.cos(theta), np.sin(theta)])
        # radius scaled
        r = scale * np.sqrt(vals)
        # transform
        pt = c + (vecs @ (r * unit))
        pts.append(pt)
    return np.array(pts)

def mvee_vertices(c, Sigma):
    vals, vecs = np.linalg.eigh(Sigma)
    verts = []
    for i in range(2):
        v = vecs[:, i]
        verts.append(c + np.sqrt(vals[i]) * v)
        verts.append(c - np.sqrt(vals[i]) * v)
    return np.array(verts)

# -----------------------------
# Visualization pipeline
# -----------------------------
def visualize_method(X3d, labels, method_name, save_path):
    # Step 2: global PCA to 2D
    pca = PCA(n_components=2, random_state=0)
    X2d = pca.fit_transform(X3d)

    k = labels.max() + 1
    # Choose one cluster (largest) for hull + MVEE
    sizes = [np.sum(labels == i) for i in range(k)]
    target_cluster = int(np.argmax(sizes))
    cluster_points = X2d[labels == target_cluster]

    # Convex hull (if enough pts)
    hull = None
    if len(cluster_points) >= 3:
        try:
            hull = ConvexHull(cluster_points)
        except Exception:
            hull = None

    # MVEE
    c, Sigma = compute_mvee(cluster_points)
    ellipse_pts = ellipse_poly(c, Sigma, scale=2.0)
    verts = mvee_vertices(c, Sigma)

    # Carathéodory union set
    r = min(2, np.linalg.matrix_rank(cluster_points))
    car_idx_union = set()
    for v in verts:
        idxs = caratheodory_set(v, cluster_points, r)
        for j in idxs:
            car_idx_union.add(np.where(labels == target_cluster)[0][j])

    selected_mask = np.zeros(len(X2d), dtype=bool)
    selected_mask[list(car_idx_union)] = True

    # Figure with 5 panels
    fig, axes = plt.subplots(1, 5, figsize=(18, 3))
    fig.suptitle(f"Pipeline ({method_name})", fontsize=12)

    # (i) 3D scatter colored by cluster
    ax = axes[0]
    # Project 3d to pseudo 2d for display or use first 2 dims
    ax.scatter(X3d[:, 0], X3d[:, 1], c=labels, cmap='tab10', s=30, edgecolors='k')
    ax.set_title('(i) 3D dữ liệu (màu cụm)')
    ax.set_xlabel('x'); ax.set_ylabel('y')

    # (ii) 2D PCA projection
    ax = axes[1]
    ax.scatter(X2d[:, 0], X2d[:, 1], c=labels, cmap='tab10', s=30, edgecolors='k')
    ax.set_title('(ii) Chiếu PCA 2D')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

    # (iii) Convex hull cụm chọn
    ax = axes[2]
    ax.scatter(cluster_points[:, 0], cluster_points[:, 1], color='red', s=35)
    if hull is not None:
        for simplex in hull.simplices:
            ax.plot(cluster_points[simplex, 0], cluster_points[simplex, 1], 'b-')
    ax.set_title('(iii) Bao lồi cụm')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

    # (iv) MVEE + vertices
    ax = axes[3]
    ax.scatter(cluster_points[:, 0], cluster_points[:, 1], color='red', s=25)
    ax.plot(ellipse_pts[:, 0], ellipse_pts[:, 1], 'g-', lw=1.2)
    ax.scatter(verts[:, 0], verts[:, 1], color='magenta', s=60, marker='X')
    ax.set_title('(iv) Ellipsoid & vertices')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

    # (v) Carathéodory coreset
    ax = axes[4]
    ax.scatter(X2d[:, 0], X2d[:, 1], c=labels, cmap='tab10', s=20, alpha=0.4)
    ax.scatter(X2d[selected_mask, 0], X2d[selected_mask, 1],
               color='black', s=80, marker='*', label='Coreset')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_title('(v) Coreset (Carathéodory)')

    plt.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)

def run_all():
    X3d = generate_points(n=40, seed=7)
    # PCA 2D for clustering base
    p2d = PCA(n_components=2, random_state=0).fit_transform(X3d)
    k_est = max(3, len(X3d) // (2 * 4))  # heuristic
    # Distance-based
    dist_labels = distance_based_clustering_2d(p2d, k_est)
    visualize_method(X3d, dist_labels, "Distance-Based", "distance_based_pipeline.png")

    # KMeans
    kmeans_labels = KMeans(n_clusters=k_est, random_state=0, n_init="auto").fit_predict(p2d)
    visualize_method(X3d, kmeans_labels, "KMeans", "kmeans_pipeline.png")

    # KMedoids (optional)
    if HAS_KMEDOIDS:
        kmed_labels = KMedoids(n_clusters=k_est, random_state=0).fit_predict(p2d)
    else:
        # Simple PAM fallback (greedy swap)
        kmed_labels = kmeans_labels  # reuse as fallback
    visualize_method(X3d, kmed_labels, "KMedoids/PAM", "kmedoids_pipeline.png")

if __name__ == "__main__":
    run_all()
    print("Generated: distance_based_pipeline.png, kmeans_pipeline.png, kmedoids_pipeline.png")

Generated: distance_based_pipeline.png, kmeans_pipeline.png, kmedoids_pipeline.png


In [2]:
# ...existing code...

def visualize_distance_based(X3d, labels, save_path):
    pca = PCA(n_components=2, random_state=0)
    X2d = pca.fit_transform(X3d)

    k = labels.max() + 1
    sizes = [np.sum(labels == i) for i in range(k)]
    target_cluster = int(np.argmax(sizes))
    cluster_points = X2d[labels == target_cluster]

    hull = None
    if len(cluster_points) >= 3:
        try:
            hull = ConvexHull(cluster_points)
        except Exception:
            hull = None

    c, Sigma = compute_mvee(cluster_points)
    ellipse_pts = ellipse_poly(c, Sigma, scale=2.0)
    verts = mvee_vertices(c, Sigma)

    r = min(2, np.linalg.matrix_rank(cluster_points))
    car_idx_union = set()
    for v in verts:
        idxs = caratheodory_set(v, cluster_points, r)
        for j in idxs:
            car_idx_union.add(np.where(labels == target_cluster)[0][j])

    selected_mask = np.zeros(len(X2d), dtype=bool)
    selected_mask[list(car_idx_union)] = True

    fig, axes = plt.subplots(1, 5, figsize=(18, 3))
    fig.suptitle("Pipeline (Distance-Based)", fontsize=12)

    # (i) 3D (dùng 2 trục đầu để nhìn phẳng)
    ax = axes[0]
    ax.scatter(X3d[:,0], X3d[:,1], c=labels, cmap='tab10', s=30, edgecolors='k')
    ax.set_title('(i) 3D dữ liệu')
    ax.set_xlabel('x'); ax.set_ylabel('y')

    # (ii) PCA 2D
    ax = axes[1]
    ax.scatter(X2d[:,0], X2d[:,1], c=labels, cmap='tab10', s=30, edgecolors='k')
    ax.set_title('(ii) PCA 2D')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

    # (iii) Tô màu toàn bộ cụm + bao lồi cụm chọn
    ax = axes[2]
    ax.scatter(X2d[:,0], X2d[:,1], c=labels, cmap='tab10', s=28, edgecolors='k', alpha=0.85)
    if hull is not None:
        for simplex in hull.simplices:
            ax.plot(cluster_points[simplex,0], cluster_points[simplex,1], 'b-')
    ax.set_title('(iii) Các cụm & Bao lồi cụm lớn nhất')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

    # (iv) Ellipsoid & vertices
    ax = axes[3]
    ax.scatter(cluster_points[:,0], cluster_points[:,1], color='red', s=25, edgecolors='k')
    ax.plot(ellipse_pts[:,0], ellipse_pts[:,1], 'g-', lw=1.2)
    ax.scatter(verts[:,0], verts[:,1], color='magenta', s=70, marker='X')
    ax.set_title('(iv) MVEE + vertices')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

    # (v) Coreset
    ax = axes[4]
    ax.scatter(X2d[:,0], X2d[:,1], c=labels, cmap='tab10', s=22, alpha=0.35)
    ax.scatter(X2d[selected_mask,0], X2d[selected_mask,1],
               color='black', s=85, marker='*', label='Coreset')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_title('(v) Coreset (Carathéodory)')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

    plt.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)

def run_distance_based():
    X3d = generate_points(n=40, seed=11)
    p2d = PCA(n_components=2, random_state=0).fit_transform(X3d)
    k_est = max(3, len(X3d) // (2 * 4))
    labels = distance_based_clustering_2d(p2d, k_est)
    visualize_distance_based(X3d, labels, "distance_based_pipeline.png")
    print("Saved distance_based_pipeline.png")

if __name__ == "__main__":
    run_distance_based()

# ...existing code...

Saved distance_based_pipeline.png
